## Exploratory Data Analysis

### Import Libraries

In [1]:
import sys, os
import polars as pl
import polars.selectors as pol_sel

### Show Python & Library Versions

In [2]:
l = 8
r = 12

print("Python".rjust(l), ":", sys.version[0:6].ljust(r))
print("Polars".rjust(l), ":", pl.__version__.ljust(r))

  Python : 3.11.4      
  Polars : 1.12.0      


## Exploratory Data Analysis with Polars 

### Load CSV File into Polars DataFrame

In [ ]:
# Path to the folder containing the files
folder_path = "data-loader/data"

# List all files in the folder
files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]  # Modify this if using other formats

# Function to load and dynamically cast numeric columns to Float64
def load_and_cast(file_path):
    # Read CSV file
    df = pl.read_csv(file_path)
    
    # Identify numeric columns (both Float64 and Int64)
    numeric_columns = [col for col, dtype in zip(df.columns, df.dtypes) if dtype in [pl.Float64, pl.Int64]]
    
    # Cast these numeric columns to Float64
    df = df.with_columns([
        pl.col(col).cast(pl.Float64) for col in numeric_columns
    ])
    
    return df

# Load the CSV files and cast numeric columns
dfs = [load_and_cast(os.path.join(folder_path, file)) for file in files]

# Concatenate the DataFrames
df = pl.concat(dfs)

In [4]:
df

transaction_id,transaction_timestamp,vehicle_type,fastag_id,toll_booth_id,lane_type,vehicle_dims,transaction_amount,amount_paid,geo_location,vehicle_speed,vehicle_plate_number,fraud_indicator
f64,str,str,str,str,str,str,f64,f64,str,f64,str,str
1.0,"""1/6/2023 11:20""","""Bus ""","""FTG-001-ABC-121""","""A-101""","""Express""","""Large""",350.0,120.0,"""13.059816123454882, 77.7706866…",65.0,"""KA11AB1234""","""Fraud"""
2.0,"""1/7/2023 14:55""","""Car""","""FTG-002-XYZ-451""","""B-102""","""Regular""","""Small""",120.0,100.0,"""13.059816123454882, 77.7706866…",78.0,"""KA66CD5678""","""Fraud"""
3.0,"""1/8/2023 18:25""","""Motorcycle""",null,"""D-104""","""Regular""","""Small""",0.0,0.0,"""13.059816123454882, 77.7706866…",53.0,"""KA88EF9012""","""Not Fraud"""
4.0,"""1/9/2023 2:05""","""Truck""","""FTG-044-LMN-322""","""C-103""","""Regular""","""Large""",350.0,120.0,"""13.059816123454882, 77.7706866…",92.0,"""KA11GH3456""","""Fraud"""
5.0,"""1/10/2023 6:35""","""Van""","""FTG-505-DEF-652""","""B-102""","""Express""","""Medium""",140.0,100.0,"""13.059816123454882, 77.7706866…",60.0,"""KA44IJ6789""","""Fraud"""
…,…,…,…,…,…,…,…,…,…,…,…,…
4996.0,"""1/1/2023 22:18""","""Truck""","""FTG-445-EDC-765""","""C-103""","""Regular""","""Large""",330.0,330.0,"""13.21331620748757, 77.55413526…",81.0,"""KA74ST0123""","""Not Fraud"""
4997.0,"""1/17/2023 13:43""","""Van""","""FTG-446-LMK-432""","""B-102""","""Express""","""Medium""",125.0,125.0,"""13.21331620748757, 77.55413526…",64.0,"""KA38UV3456""","""Not Fraud"""
4998.0,"""2/5/2023 5:08""","""Sedan""","""FTG-447-PLN-109""","""A-101""","""Regular""","""Medium""",115.0,115.0,"""13.21331620748757, 77.55413526…",93.0,"""KA33WX6789""","""Not Fraud"""


In [5]:
column_data_types = zip(df.columns, df.dtypes)

for column, dtype in column_data_types:
    print(f"{column}:\t\t{dtype}")

transaction_id:		Float64
transaction_timestamp:		String
vehicle_type:		String
fastag_id:		String
toll_booth_id:		String
lane_type:		String
vehicle_dims:		String
transaction_amount:		Float64
amount_paid:		Float64
geo_location:		String
vehicle_speed:		Float64
vehicle_plate_number:		String
fraud_indicator:		String


### Display First Few Rows to Understand Structure of Data

In [6]:
print(df.head())

shape: (5, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ transacti ┆ transacti ┆ vehicle_t ┆ fastag_id ┆ … ┆ geo_locat ┆ vehicle_s ┆ vehicle_p ┆ fraud_in │
│ on_id     ┆ on_timest ┆ ype       ┆ ---       ┆   ┆ ion       ┆ peed      ┆ late_numb ┆ dicator  │
│ ---       ┆ amp       ┆ ---       ┆ str       ┆   ┆ ---       ┆ ---       ┆ er        ┆ ---      │
│ f64       ┆ ---       ┆ str       ┆           ┆   ┆ str       ┆ f64       ┆ ---       ┆ str      │
│           ┆ str       ┆           ┆           ┆   ┆           ┆           ┆ str       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1.0       ┆ 1/6/2023  ┆ Bus       ┆ FTG-001-A ┆ … ┆ 13.059816 ┆ 65.0      ┆ KA11AB123 ┆ Fraud    │
│           ┆ 11:20     ┆           ┆ BC-121    ┆   ┆ 123454882 ┆           ┆ 4         ┆          │
│           ┆           ┆           ┆           ┆   ┆ , 77.7706 ┆           

### Retrieve Basic Information About DataFrame

In [7]:
print(df.shape)
print(df.dtypes)

(5000, 13)
[Float64, String, String, String, String, String, String, Float64, Float64, String, Float64, String, String]


### Display Summary Statistics for All Columns

In [8]:
summary = df.describe()
print(summary)

shape: (9, 14)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ transacti ┆ transacti ┆ vehicle_t ┆ … ┆ geo_locat ┆ vehicle_s ┆ vehicle_p ┆ fraud_in │
│ ---       ┆ on_id     ┆ on_timest ┆ ype       ┆   ┆ ion       ┆ peed      ┆ late_numb ┆ dicator  │
│ str       ┆ ---       ┆ amp       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ er        ┆ ---      │
│           ┆ f64       ┆ ---       ┆ str       ┆   ┆ str       ┆ f64       ┆ ---       ┆ str      │
│           ┆           ┆ str       ┆           ┆   ┆           ┆           ┆ str       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ count     ┆ 5000.0    ┆ 5000      ┆ 5000      ┆ … ┆ 5000      ┆ 5000.0    ┆ 5000      ┆ 5000     │
│ null_coun ┆ 0.0       ┆ 0         ┆ 0         ┆ … ┆ 0         ┆ 0.0       ┆ 0         ┆ 0        │
│ t         ┆           ┆           ┆           ┆   ┆           ┆           

### Count the Number of Null Values in Each Column

In [9]:
for col in df.columns:
    print(f"{col} : {df[col].is_null().sum()}")

transaction_id : 0
transaction_timestamp : 0
vehicle_type : 0
fastag_id : 549
toll_booth_id : 0
lane_type : 0
vehicle_dims : 0
transaction_amount : 0
amount_paid : 0
geo_location : 0
vehicle_speed : 0
vehicle_plate_number : 0
fraud_indicator : 0


### Find Longest Text Length in Each Column

In [10]:
# Create an empty list to store max lengths for each string column
longest_text_lengths = []

# Loop through the columns to check for string columns
string_columns = [col for col in df.columns if df[col].dtype == pl.Utf8]

max_lengths = {}
for col in string_columns:
    max_length = df.select(pl.col(col).str.len_chars().max()).to_numpy()[0, 0]
    max_lengths[col] = max_length

df_max_lengths = pl.DataFrame(max_lengths)

df_max_lengths

transaction_timestamp,vehicle_type,fastag_id,toll_booth_id,lane_type,vehicle_dims,geo_location,vehicle_plate_number,fraud_indicator
u32,u32,u32,u32,u32,u32,u32,u32,u32
16,10,16,5,7,6,37,10,9


### Retrieve Data Types of All Columns

In [16]:
print(f"Column data types:\n{df.dtypes}")

Column data types:
[Float64, String, String, String, String, String, String, Float64, Float64, String, Float64, String, String]


### Count Unique Values in Each Column

In [12]:
all_columns = [col for col in df.columns]

for col in all_columns:
    unique_counts = df[col].n_unique()
    print(f"Unique values in {col} :".rjust(48), f"{unique_counts}".ljust(6))

               Unique values in transaction_id : 5000  
        Unique values in transaction_timestamp : 4423  
                 Unique values in vehicle_type : 7     
                    Unique values in fastag_id : 4452  
                Unique values in toll_booth_id : 6     
                    Unique values in lane_type : 2     
                 Unique values in vehicle_dims : 3     
           Unique values in transaction_amount : 20    
                  Unique values in amount_paid : 23    
                 Unique values in geo_location : 5     
                Unique values in vehicle_speed : 85    
         Unique values in vehicle_plate_number : 5000  
              Unique values in fraud_indicator : 2     


### Retrieve Unique Values in Column If There Are Fewer Than 100

In [13]:
for col in df.columns:
    # Get unique values for the column
    unique_values = df[col].unique().to_list()  # Convert to a list
    if len(unique_values) < 100:
        print(f"'{col}' {len(unique_values)}: {unique_values}")

'vehicle_type' 7: ['Motorcycle', 'Car', 'Truck', 'SUV', 'Van', 'Bus ', 'Sedan']
'toll_booth_id' 6: ['D-106', 'D-104', 'D-105', 'A-101', 'B-102', 'C-103']
'lane_type' 2: ['Express', 'Regular']
'vehicle_dims' 3: ['Medium', 'Large', 'Small']
'transaction_amount' 20: [0.0, 60.0, 70.0, 90.0, 100.0, 110.0, 115.0, 120.0, 125.0, 130.0, 140.0, 145.0, 150.0, 160.0, 180.0, 290.0, 300.0, 330.0, 340.0, 350.0]
'amount_paid' 23: [0.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0, 110.0, 115.0, 120.0, 125.0, 130.0, 140.0, 145.0, 150.0, 160.0, 180.0, 190.0, 290.0, 300.0, 330.0, 340.0, 350.0]
'geo_location' 5: ['12.84197701525119, 77.67547528176169', '13.059816123454882, 77.77068662374292', '13.21331620748757, 77.55413526894684', '12.936687032945434, 77.53113977439017', '13.042660878688794, 77.47580097259879']
'vehicle_speed' 85: [10.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0, 48.0, 49.0

### Check Distribution of Numerical Columns

In [14]:
numerical_cols = [item for item in all_columns if item not in string_columns]
numerical_cols = [item for item in numerical_cols if item not in ['id']]

numerical_cols

for col in numerical_cols:
    distribution = df.select(col).describe()
    print(col)
    print(distribution, '\n\n')

transaction_id
shape: (9, 2)
┌────────────┬────────────────┐
│ statistic  ┆ transaction_id │
│ ---        ┆ ---            │
│ str        ┆ f64            │
╞════════════╪════════════════╡
│ count      ┆ 5000.0         │
│ null_count ┆ 0.0            │
│ mean       ┆ 2500.5         │
│ std        ┆ 1443.520003    │
│ min        ┆ 1.0            │
│ 25%        ┆ 1251.0         │
│ 50%        ┆ 2501.0         │
│ 75%        ┆ 3750.0         │
│ max        ┆ 5000.0         │
└────────────┴────────────────┘ 


transaction_amount
shape: (9, 2)
┌────────────┬────────────────────┐
│ statistic  ┆ transaction_amount │
│ ---        ┆ ---                │
│ str        ┆ f64                │
╞════════════╪════════════════════╡
│ count      ┆ 5000.0             │
│ null_count ┆ 0.0                │
│ mean       ┆ 161.062            │
│ std        ┆ 112.44995          │
│ min        ┆ 0.0                │
│ 25%        ┆ 100.0              │
│ 50%        ┆ 130.0              │
│ 75%        ┆ 290.0   